# Module 3 — Stacks and Queues

This is the worked reference notebook: run live in lecture, fully solved.
The version students receive with TODOs in place of the solved parts is
`assignments/pds/a2-stacks-queues/starter/stacks_queues.py`.

## 1. `Stack`, built on the dynamic array (Lecture 1)

In [1]:
class Stack:
    def __init__(self):
        self._items = []

    def push(self, x):
        self._items.append(x)

    def pop(self):
        if not self._items:
            raise IndexError("pop from an empty stack")
        return self._items.pop()

    def peek(self):
        if not self._items:
            raise IndexError("peek at an empty stack")
        return self._items[-1]

    def is_empty(self):
        return len(self._items) == 0

s = Stack()
s.push(1); s.push(2); s.push(3)
assert s.pop() == 3
assert s.peek() == 2
assert not s.is_empty()
print("Stack checks passed")

Stack checks passed


## 2. Balanced parentheses (Lecture 1)

In [2]:
def is_balanced(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = Stack()
    for ch in s:
        if ch in "([{":
            stack.push(ch)
        elif ch in ")]}":
            if stack.is_empty() or stack.pop() != pairs[ch]:
                return False
    return stack.is_empty()

assert is_balanced("([{}])") == True
assert is_balanced("([)]") == False
assert is_balanced("(a + [b * (c - d)]) / e") == True
print("Balanced-parentheses checks passed")

Balanced-parentheses checks passed


## 3. The naive queue's hidden O(n) trap (Lecture 2)

In [3]:
import time

def time_naive_dequeue(n):
    q = list(range(n))
    start = time.perf_counter()
    for _ in range(n):
        q.pop(0)
    return time.perf_counter() - start

for n in (2_000, 4_000, 8_000):
    t = time_naive_dequeue(n)
    print(f"n={n:>6}  naive dequeue total time={t:.4f}s")

n=  2000  naive dequeue total time=0.0056s
n=  4000  naive dequeue total time=0.0285s
n=  8000  naive dequeue total time=0.1191s


The timing should grow noticeably faster than linearly as `n` doubles —
evidence of the O(n²) total cost Lecture 2 predicted for the naive,
`pop(0)`-based queue.

## 4. `deque`-backed `Queue`, fixed (Lecture 2)

In [4]:
from collections import deque

class Queue:
    def __init__(self):
        self._items = deque()

    def enqueue(self, x):
        self._items.append(x)

    def dequeue(self):
        if not self._items:
            raise IndexError("dequeue from an empty queue")
        return self._items.popleft()

    def is_empty(self):
        return len(self._items) == 0

jobs = Queue()
jobs.enqueue("report.pdf")
jobs.enqueue("photo.png")
jobs.enqueue("invoice.pdf")
assert jobs.dequeue() == "report.pdf"
assert jobs.dequeue() == "photo.png"
assert jobs.dequeue() == "invoice.pdf"
print("deque-backed Queue checks passed, FIFO order confirmed")

deque-backed Queue checks passed, FIFO order confirmed


## 5. `CircularQueue`, fixed capacity (Lecture 3)

In [5]:
class CircularQueue:
    def __init__(self, capacity):
        self.capacity = capacity
        self._data = [None] * capacity
        self.front = 0
        self.back = 0
        self.count = 0

    def is_empty(self):
        return self.count == 0

    def is_full(self):
        return self.count == self.capacity

    def enqueue(self, x):
        if self.is_full():
            raise OverflowError("circular queue is full")
        self._data[self.back] = x
        self.back = (self.back + 1) % self.capacity
        self.count += 1

    def dequeue(self):
        if self.is_empty():
            raise IndexError("dequeue from an empty queue")
        x = self._data[self.front]
        self.front = (self.front + 1) % self.capacity
        self.count -= 1
        return x

cq = CircularQueue(3)
cq.enqueue('A'); cq.enqueue('B'); cq.enqueue('C')
assert cq.is_full()
assert cq.dequeue() == 'A'
cq.enqueue('D')   # wraps into slot 0, previously held 'A'
assert cq.dequeue() == 'B'
assert cq.dequeue() == 'C'
assert cq.dequeue() == 'D'
assert cq.is_empty()
print("CircularQueue checks passed, matching Lecture 3's worked trace")

CircularQueue checks passed, matching Lecture 3's worked trace


## 6. Postfix evaluation and the shunting-yard algorithm (Lecture 4)

In [6]:
def evaluate_postfix(tokens):
    stack = []
    for tok in tokens:
        if tok in "+-*/":
            b = stack.pop()
            a = stack.pop()
            if tok == '+': stack.append(a + b)
            elif tok == '-': stack.append(a - b)
            elif tok == '*': stack.append(a * b)
            elif tok == '/': stack.append(a / b)
        else:
            stack.append(float(tok))
    return stack.pop()

PRECEDENCE = {'+': 1, '-': 1, '*': 2, '/': 2}

def infix_to_postfix(tokens):
    output = []
    ops = []
    for tok in tokens:
        if tok not in PRECEDENCE:
            output.append(tok)
        else:
            while (ops and ops[-1] in PRECEDENCE
                   and PRECEDENCE[ops[-1]] >= PRECEDENCE[tok]):
                output.append(ops.pop())
            ops.append(tok)
    while ops:
        output.append(ops.pop())
    return output

def calculate(infix_tokens):
    return evaluate_postfix(infix_to_postfix(infix_tokens))

assert evaluate_postfix(['3', '4', '2', '*', '+']) == 11.0
assert infix_to_postfix(['3', '+', '4', '*', '2']) == ['3', '4', '2', '*', '+']
assert calculate(['3', '+', '4', '*', '2']) == 11.0
assert calculate(['10', '-', '2', '*', '3']) == 4.0
print("Postfix evaluation and shunting-yard checks passed")

Postfix evaluation and shunting-yard checks passed


## 7. MTech addition — the monotonic stack (Lecture 4)

In [7]:
def next_greater_elements(arr):
    result = [-1] * len(arr)
    stack = []
    for i, x in enumerate(arr):
        while stack and arr[stack[-1]] < x:
            result[stack.pop()] = x
        stack.append(i)
    return result

assert next_greater_elements([2, 1, 2, 4, 3]) == [4, 2, 4, -1, -1]
print("Monotonic stack (next greater element) check passed")

Monotonic stack (next greater element) check passed
